# Breast Cancer Diagnostic: Prediction using Machine Learning

**Intern ID:** `<CITS9882>`  
**Full Name:** `<pasupuleti krishnakamalika>`  
**No. of Weeks:** `<12 weeks>`  
**Project Name:** Breast Cancer Diagnostic  
**Project Scope:** Data cleaning, EDA, visualization, and building/comparing ML models to classify tumors as **Malignant (M)** or **Benign (B)**.

**Dataset:** Breast Cancer Wisconsin (Diagnostic): 569 samples, 30 numeric features computed from digitized images of fine needle aspirates (FNA) of breast masses.

## 1. Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42

# Folders for outputs (these match the GitHub repo structure)
os.makedirs("images", exist_ok=True)
os.makedirs("models", exist_ok=True)

## 2. Load the Dataset
If you downloaded the Kaggle CSV, put it at `data/data.csv`. If the file is not found, the notebook falls back to scikit-learn's built-in copy of the same dataset.

In [ ]:
if os.path.exists("data/data.csv"):
    df = pd.read_csv("data/data.csv")
    print("Loaded Kaggle CSV from data/data.csv")
else:
    bunch = load_breast_cancer(as_frame=True)
    df = bunch.frame.copy()
    # In scikit-learn, target 0 = malignant and 1 = benign. Convert to M/B to match Kaggle.
    df["diagnosis"] = df["target"].map({0: "M", 1: "B"})
    df = df.drop(columns="target")
    print("Loaded dataset from scikit-learn")

print("Shape:", df.shape)
df.head()

## 3. Data Cleaning

In [ ]:
df.info()

In [ ]:
# Missing values and duplicates
print("Total missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

In [ ]:
# Drop columns that carry no information (present only in the Kaggle version)
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])

# Remove duplicates if any
df = df.drop_duplicates().reset_index(drop=True)

# Encode target: Malignant = 1, Benign = 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

print("Clean shape:", df.shape)
print(df["diagnosis"].value_counts().rename({0: "Benign (0)", 1: "Malignant (1)"}))

## 4. Exploratory Data Analysis (EDA)

### 4.1 Statistical summary

In [ ]:
df.describe().T.round(3)

### 4.2 Class distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x="diagnosis", data=df, palette=["#2a9d8f", "#e63946"], ax=ax[0])
ax[0].set_xticklabels(["Benign", "Malignant"])
ax[0].set_title("Diagnosis Count")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom", fontweight="bold")

df["diagnosis"].value_counts().plot.pie(autopct="%1.1f%%", labels=["Benign", "Malignant"],
                                        colors=["#2a9d8f", "#e63946"], ax=ax[1], startangle=90)
ax[1].set_ylabel("")
ax[1].set_title("Diagnosis Share")

plt.tight_layout()
plt.savefig("images/01_class_distribution.png", dpi=150)
plt.show()

**Observation:** The dataset is moderately imbalanced (about 63% benign, 37% malignant). That is why accuracy alone is not enough: we also track **recall** for the malignant class, since missing a cancer case is the costliest error.

### 4.3 Feature distributions by diagnosis

In [ ]:
mean_features = [c for c in df.columns if c.endswith("_mean") or c.startswith("mean ")]
if not mean_features:
    mean_features = list(df.columns[1:11])

plot_df = df.copy()
plot_df["Diagnosis"] = plot_df["diagnosis"].map({0: "Benign", 1: "Malignant"})

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.ravel(), mean_features):
    sns.histplot(data=plot_df, x=col, hue="Diagnosis", kde=True, palette=["#2a9d8f", "#e63946"], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("images/02_feature_distributions.png", dpi=150)
plt.show()

### 4.4 Box plots of key features

In [ ]:
key_candidates = ["radius_mean", "texture_mean", "area_mean", "concavity_mean",
                  "mean radius", "mean texture", "mean area", "mean concavity"]
key_features = [c for c in key_candidates if c in df.columns]

fig, axes = plt.subplots(1, len(key_features), figsize=(5 * len(key_features), 5))
axes = np.atleast_1d(axes)
for ax, col in zip(axes, key_features):
    sns.boxplot(data=plot_df, x="Diagnosis", y=col, palette=["#2a9d8f", "#e63946"], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("images/03_boxplots_key_features.png", dpi=150)
plt.show()

### 4.5 Correlation analysis

In [ ]:
corr = df.corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, linewidths=0.3, cbar_kws={"shrink": 0.7})
plt.title("Correlation Heatmap of All Features", fontsize=16)
plt.tight_layout()
plt.savefig("images/04_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Features most correlated with the diagnosis
target_corr = corr["diagnosis"].drop("diagnosis").sort_values(ascending=False)

plt.figure(figsize=(8, 9))
sns.barplot(x=target_corr.values, y=target_corr.index, palette="coolwarm_r")
plt.title("Correlation of Each Feature with Diagnosis (Malignant = 1)")
plt.xlabel("Correlation")
plt.tight_layout()
plt.savefig("images/05_target_correlation.png", dpi=150)
plt.show()

print("Top 5 positively correlated features:")
print(target_corr.head(5).round(3))

**Observation:** Size and shape features of the worst, mean and perimeter/radius/area groups (e.g. `concave points`, `perimeter`, `radius`) show the strongest positive correlation with malignancy. Many features are also highly correlated with each other (radius, perimeter and area), which is expected, since they measure related things.

## 5. Data Preparation

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

# 80/20 split, stratified so both sets keep the same benign/malignant ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Malignant share in train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

Scaling is done **inside a Pipeline** so the scaler is fit only on training data. This avoids data leakage into the test set.

## 6. Build and Compare Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN":                 KNeighborsClassifier(n_neighbors=7),
    "SVM (RBF)":           SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=RANDOM_STATE),
}

pipelines = {name: Pipeline([("scaler", StandardScaler()), ("clf", m)]) for name, m in models.items()}

# 5-fold stratified cross-validation on the training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")
    cv_results[name] = scores
    print(f"{name:20s} CV recall: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 7. Train and Evaluate on the Test Set

In [ ]:
rows = []
fitted = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall":    recall_score(y_test, pred),
        "F1-score":  f1_score(y_test, pred),
        "ROC-AUC":   roc_auc_score(y_test, proba),
    })

results = pd.DataFrame(rows).set_index("Model").round(4).sort_values("F1-score", ascending=False)
results

In [ ]:
ax = results.plot(kind="bar", figsize=(12, 5), ylim=(0.85, 1.01), colormap="viridis")
plt.title("Model Comparison on Test Set")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/06_model_comparison.png", dpi=150)
plt.show()

### 7.1 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(fitted), figsize=(5 * len(fitted), 4))
for ax, (name, pipe) in zip(axes, fitted.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("images/07_confusion_matrices.png", dpi=150)
plt.show()

### 7.2 ROC curves

In [ ]:
plt.figure(figsize=(7, 6))
for name, pipe in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("images/08_roc_curves.png", dpi=150)
plt.show()

## 8. Hyperparameter Tuning (best model)

In [ ]:
# Tune an SVM with grid search, using recall as the scoring metric (missing cancer is the costliest error)
param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.01, 0.001],
    "clf__kernel": ["rbf", "linear"],
}
svm_pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))])

grid = GridSearchCV(svm_pipe, param_grid, cv=cv, scoring="recall", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV recall: %.4f" % grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest-set classification report (tuned SVM):")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False,
            xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.title("Tuned SVM: Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("images/09_tuned_svm_confusion_matrix.png", dpi=150)
plt.show()

## 9. Feature Importance (Random Forest)

In [ ]:
rf = fitted["Random Forest"].named_steps["clf"]
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Top 10 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("images/10_feature_importance.png", dpi=150)
plt.show()

## 10. Save the Final Model and Results

In [ ]:
joblib.dump(best_model, "models/breast_cancer_svm.joblib")
results.to_csv("model_results.csv")
print("Saved: models/breast_cancer_svm.joblib and model_results.csv")

# Quick test: predict for the first 5 test samples
sample = X_test.iloc[:5]
print("Predictions:", best_model.predict(sample))
print("Actual:     ", y_test.iloc[:5].values)

## 11. Conclusion

- The dataset has **569 samples** and **30 features** with no missing values; about **37% of cases are malignant**.
- Size and shape features (`radius`, `perimeter`, `area`, `concave points`) are the strongest indicators of malignancy.
- Five models were compared using cross-validation and a held-out test set. All achieved high accuracy, with the best models reaching roughly **96-97%** accuracy on the test set.
- Because missing a malignant case is the costliest mistake, **recall** was used to tune the final **SVM** model.
- **Limitations:** small dataset, a single train/test split, and no external validation. This is a learning project and **not** a clinical diagnostic tool.
- **Future work:** try feature selection or PCA, handle class imbalance with SMOTE, and deploy the model in a Streamlit app.